In [1]:
import os
import numpy as np
import cv2

In [2]:
# read depth image
depth_scale = 0.00012498664727900177
depth_img = cv2.imread('depth.png')
dpt = depth_img[:, :, 2].astype(np.uint16) + depth_img[:, :, 1].astype(np.uint16) * 256
dpt = dpt * depth_scale

# read seg image
seg = cv2.imread('seg.png')[...,0]  # 255: fore ground, 0: background

# read intrinsics and extrinsics
K = np.load('intrinsic.npy')
print(K)

os.makedirs('../results', exist_ok=True)

[[415.69219382   0.         320.        ]
 [  0.         415.69219382 240.        ]
 [  0.           0.           1.        ]]


In [3]:
# task1: convert depth image to point cloud
def depth2pc(depth, seg, K):
    # ------------TODO---------------
    # compute point cloud from depth image
    # for-loop is not allowed!!
    h, w = depth.shape
    x, y = np.meshgrid(np.arange(w), np.arange(h))
    z = depth
    mask = seg > 0
    x = x[mask]
    y = y[mask]
    z = z[mask]

    x = (x - K[0, 2]) * z / K[0, 0] # x = (u - cx) * z / fx
    y = (y - K[1, 2]) * z / K[1, 1] # y = (v - cy) * z / fy
    pc = np.stack([x, y, z], axis=-1).reshape(-1, 3)
    # ------------TODO --------------
    return pc

partial_pc = depth2pc(dpt, seg, K)

# For debug and submission
np.savetxt('../results/pc_from_depth.txt', partial_pc)

In [4]:
# task2: compute one-way chamfer distance to the complete shape
full_pc = np.loadtxt('aligned_full_pc.txt')
from scipy.spatial import cKDTree

# -----------TODO---------------
# implement one way chamfer distance
# for each point in partial_pc, find the nearest point in full_pc and compute the distance
# partial_pc: (N, 3), full_pc: (M, 3)
tree = cKDTree(full_pc)
dists, _ = tree.query(partial_pc, k=1)
one_way_CD = dists.mean()

# -----------TODO---------------
print('one way chamfer distance: ', one_way_CD)

# For submission
np.savetxt('../results/one_way_CD.txt', np.array([one_way_CD]))

one way chamfer distance:  0.005235365614225034
